In [192]:
import sys
import pandas as pd
import numpy as np
import shap
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import os
import statistics

In [193]:
from sklearn.datasets import load_breast_cancer

def calc_shap(model, x_explain):
    explainer = shap.KernelExplainer(model.predict_proba, X_train)        
    shap_values = explainer.shap_values(x_explain)
    return shap_values, explainer

from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = MLPClassifier()
model.fit(X_train, y_train)

# Insert here the instance we want to examine
instance = 24
local = X[instance]
# from numpy import asarray
# input = asarray([local])

In [194]:
# get the shapley values for the whole dataset
shap_values, explainer = calc_shap(model, local)   

Using 426 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 
The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the prev

In [197]:
# ----------- Extract local predictions through the loops -----------
def grab_predictions(instance,iteration):
    prediction = model.predict(instance)
    preds = []
    if prediction.item(0) == 1:
        preds.append("Iteration Number %d: " %iteration +"Breast cancer detected :(")
    else:
        preds.append("Iteration Number %d: " %iteration +"You are okay :)")
    preds = pd.DataFrame(preds)  
    preds.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\L_E_SHAP\predictions.txt', header=None, index=None, sep=' ', mode='a')


In [ ]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\L_E_SHAP\features.txt','r') as file:
    data = file.readlines()
file.close()

In [ ]:
values = []
features = []
experiments = 10
for i in range(10):
    for j in range(5):
        x = data[i*8+j].split()
        temp = (x[2].split('"'))
        values.append(x[3])
        features.append(x[1]+" "+str(temp[0]))

In [ ]:
# Make dataframe with the shapley values of the features for each run  
values = np.array(values)
sub_arrays = np.hsplit(values, experiments)
df = pd.DataFrame(sub_arrays)
for i in range(experiments):
    df.index = ['Run_%d'%(i+1) for i in range(experiments)]
df.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

# Make dataframe with the feature names (ranking of the features at each experiment)
features = np.array(features)
sub_arrays2 = np.hsplit(features, experiments)
df2 = pd.DataFrame(sub_arrays2)
for i in range(experiments):
    df2.index = ['Run_%d'%(i+1) for i in range(experiments)]
df2.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']
df2 = df2.applymap(lambda x: x.replace('"', ''))

In [ ]:
# Gather here the unique features
uniqueValues = (df2['1st Feature'].append(df2['2nd Feature'].append(df2['3rd Feature'].append(df2['4th Feature'].append(df2['5th Feature']))))).unique()

df2 = df2.astype(str)
print("\nOur dataframes are:\n\n",df,"\n\n",df2,"\n")  
print("\nDescribe df with the feature names:\n\n", df2.describe())
print("\n..........................................\n")  


Our dataframes are:

                    1st Feature             2nd Feature  \
Run_1    3.885780586188048e-15   4.718447854656915e-16   
Run_2    2.220446049250313e-16   5.551115123125783e-17   
Run_3   3.1086244689504383e-15   7.771561172376096e-16   
Run_4    4.440892098500626e-16   2.220446049250313e-16   
Run_5   4.3298697960381105e-15   6.245004513516506e-16   
Run_6   3.6637359812630166e-15  1.1102230246251565e-15   
Run_7    5.551115123125783e-16  2.8449465006019636e-16   
Run_8   1.1102230246251565e-16    9.71445146547012e-17   
Run_9    4.218847493575595e-15   6.661338147750939e-16   
Run_10              dimension"   2.220446049250313e-16   

                   3rd Feature             4th Feature             5th Feature  
Run_1    4.163336342344337e-16   3.885780586188048e-16   3.608224830031759e-16  
Run_2    5.551115123125783e-17   5.551115123125783e-17   5.551115123125783e-17  
Run_3    6.661338147750939e-16   6.661338147750939e-16   6.106226635438361e-16  
Run_4   1.6696

In [ ]:
from decimal import Decimal
x = df.iloc[9][0]
d = Decimal(x)
print(float(d))

0.0


In [ ]:
def is_float(value):
    try:
        float(value)
        return True
    except:
        return False

for i in range(df.shape[0]): #iterate over rows
    for j in range(df.shape[1]): #iterate over columns
        if is_float(df.iloc[i][j]):
            df.iloc[i][j] = Decimal(df.iloc[i][j])
        else:
            df.iloc[i][j] = 0

In [ ]:
df1 = df.astype(float)